In [1]:
import numpy as np
import pandas as pd
import ast

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [8]:
movies = movies.merge(credits,on ='title')

In [9]:
movies.dropna(inplace = True)

In [10]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [11]:
def convert(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [12]:
movies['genres'] = movies['genres'].apply(convert)

In [13]:
movies['keywords'] = movies['keywords'].apply(convert)

In [18]:
def convert3(obj):
    l = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            l.append(i['name'])
            counter +=1
        else:
            break
    return l

In [19]:
movies['cast'] = movies['cast'].apply(convert3)

In [20]:
def fetch_director(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
        break
    return l

In [21]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [22]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [23]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [24]:
movies['tags'] = movies['overview']+ movies['genres'] + movies['keywords']+ movies['cast']+movies['crew']

In [25]:
new_df = movies[['movie_id','title','tags']]

In [26]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\praka\AppData\Local\Temp\ipykernel_25216\487797088.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))


In [27]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\praka\AppData\Local\Temp\ipykernel_25216\3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [28]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000,stop_words =  'english')

In [29]:
cv.fit_transform(new_df['tags']).toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1494, 5000))

In [30]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [31]:
cv.get_feature_names_out()

array(['00', '000', '007', ..., 'zooey', 'zooeydeschanel', 'zookeeper'],
      shape=(5000,), dtype=object)

In [32]:
import nltk

In [33]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [34]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)
    

In [35]:
new_df['tags']=new_df['tags'].apply(stem)

C:\Users\praka\AppData\Local\Temp\ipykernel_25216\3514595201.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(stem)


In [36]:
stem('in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron')

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [37]:
cv.get_feature_names_out()

array(['00', '000', '007', ..., 'zooey', 'zooeydeschanel', 'zookeeper'],
      shape=(5000,), dtype=object)

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

In [39]:
similarity = cosine_similarity(vectors)

In [40]:
sorted(list(enumerate(similarity[0])),reverse = True,key = lambda x:x[1])[1:6]

[(579, np.float64(0.22831482556870472)),
 (188, np.float64(0.22718473369882594)),
 (331, np.float64(0.223606797749979)),
 (51, np.float64(0.21764287503300347)),
 (1244, np.float64(0.2171861213815347))]

In [41]:
sorted(similarity[0],reverse=True)

[np.float64(1.0000000000000002),
 np.float64(0.22831482556870472),
 np.float64(0.22718473369882594),
 np.float64(0.223606797749979),
 np.float64(0.21764287503300347),
 np.float64(0.2171861213815347),
 np.float64(0.2053959590644373),
 np.float64(0.1978141420187361),
 np.float64(0.19364916731037088),
 np.float64(0.1897366596101028),
 np.float64(0.1897366596101028),
 np.float64(0.1863389981249825),
 np.float64(0.18526206815833915),
 np.float64(0.18446619684315546),
 np.float64(0.17541160386140586),
 np.float64(0.17320508075688773),
 np.float64(0.1728526794136782),
 np.float64(0.1688819906341803),
 np.float64(0.16878498775964426),
 np.float64(0.16878498775964426),
 np.float64(0.16770509831248423),
 np.float64(0.1651445647689541),
 np.float64(0.1651445647689541),
 np.float64(0.1649915822768611),
 np.float64(0.16010281506708454),
 np.float64(0.15811388300841894),
 np.float64(0.1559625734730109),
 np.float64(0.15504341823651058),
 np.float64(0.15504341823651058),
 np.float64(0.154982604969516

In [42]:
sorted(similarity[0][-10:-1])

[np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.02795084971874737),
 np.float64(0.031008683647302117),
 np.float64(0.03952847075210474),
 np.float64(0.055048188256318034)]

In [43]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse = True,key = lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [44]:
recommend('Avatar')

Aliens vs Predator: Requiem
Ender's Game
Battle: Los Angeles
Jupiter Ascending
Apollo 18


In [45]:
import pickle

In [46]:
pickle.dump(new_df,open('movies.pkl','wb'))

In [47]:
pickle.dump(new_df.to_dict(),open('movies_dict.pkl','wb'))

In [48]:
pickle.dump(similarity,open('similarity.pkl','wb'))